# Customer Churn

In [0]:
%run ./churn_model/utilities

In [ ]:
# https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset
# load dataset from Kaggle to Databricks Delta table

In [ ]:
import mlflow

# Set the MLflow registry to Unity Catalog
mlflow.set_registry_uri("databricks-uc") # change here for experiments
logger.info(f"MLflow registry URI set to Unity Catalog | target catalog: {TARGET_CATALOG}")

In [0]:
from mlflow.models.signature import infer_signature
from datetime import datetime

model_fully_qualified_name = f"{TARGET_CATALOG}.{config.default.schema}.{config.default.model_name}"
run_timestamp = datetime.now().strftime("%Y_%m_%d__%H_%M_%S")
run_name = f"{config.default.model_name}_{run_timestamp}"
run_description = f"Customer churn model run | environment={environment} | catalog={TARGET_CATALOG}"

# Infer the signature from a small sample. This is required for Unity Catalogue registration
sample_input_spark_dataframe = spark.table(f"{TARGET_CATALOG}.{config.default.base_table}").limit(10)
sample_input_pandas_dataframe = sample_input_spark_dataframe.toPandas()
sample_output_pandas_dataframe = ChurnModel().predict(None, sample_input_spark_dataframe).toPandas()
signature = infer_signature(sample_input_pandas_dataframe, sample_output_pandas_dataframe)

with mlflow.start_run(run_name=run_name, description=run_description) as run:

    model_info = mlflow.pyfunc.log_model(
        name=config.default.artifact_path,
        python_model=ChurnModel(),
        registered_model_name=model_fully_qualified_name,
        pip_requirements=["mlflow", "pyspark"],
        signature=signature,
    )

    version = model_info.registered_model_version

    # Tag the registered model version with relevant configuration parameters for traceability
    client = mlflow.MlflowClient()
    version_tags = {
        "learning_rate":         str(config.model_parameters.learning_rate),
        "regularisation":        str(config.model_parameters.regularisation),
        "regularisation_type":   str(config.model_parameters.regularisation_type),
        "tree_depth":            str(config.model_parameters.tree_depth),
        "n_estimators":          str(config.model_parameters.n_estimators),
        "min_samples_leaf":      str(config.model_parameters.min_samples_leaf),
        "max_features":          str(config.model_parameters.max_features),
        "early_stopping_rounds": str(config.model_parameters.early_stopping_rounds),
        "objective":             str(config.model_parameters.objective),
    }
    for tag_key, tag_value in version_tags.items():
        client.set_model_version_tag(name=model_fully_qualified_name, version=version, key=tag_key, value=tag_value)

    client.set_registered_model_alias(name=model_fully_qualified_name, alias="latest", version=version) # alias for easy access to the latest version

    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    run_url = f"https://{workspace_url}/ml/experiments/{run.info.experiment_id}/runs/{run.info.run_id}"
    all_runs_url = f"https://{workspace_url}/ml/experiments/{run.info.experiment_id}/runs/"
    uc_model_url = f"https://{workspace_url}/explore/data/models/{TARGET_CATALOG}/{config.default.schema}/{config.default.model_name}"

    print(f"Model registered : {model_fully_qualified_name}")
    print(f"Run name         : {run_name}")
    print(f"Version          : {version}")
    print(f"Run URL          : {run_url}")
    print(f"All runs         : {all_runs_url}")
    print(f"UC model versions: {uc_model_url}")
    logger.info(f"Model registered: {model_fully_qualified_name} | version={version} | run_id={run.info.run_id} | url={run_url}")

In [0]:
# Run the registered model on the input data
loaded_model = mlflow.pyfunc.load_model(f"models:/{model_fully_qualified_name}@latest")

In [ ]:
result = loaded_model.predict(spark.table(f"{TARGET_CATALOG}.{config.default.base_table}"))

In [0]:
display(result)

In [0]:
# Save the result to a Delta table in Unity Catalog
result.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_CATALOG}.{config.default.target_table}")